# Module 1: Data Loading and Augmentation Using Keras
This notebook demonstrates building custom data generators, batch processing, image augmentation, and creating validation pipelines in Keras.


In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image

base_dir = './images_dataSAT'
dir_non_agri = os.path.join(base_dir, 'class_0_non_agri')
dir_agri = os.path.join(base_dir, 'class_1_agri')
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 8

def custom_data_generator(image_paths, labels, batch_size):
    num_samples = len(image_paths)
    while True:
        for offset in range(0, num_samples, batch_size):
            batch_paths = image_paths[offset:offset+batch_size]
            batch_labels = labels[offset:offset+batch_size]
            batch_images = [np.array(Image.open(p).resize(IMAGE_SIZE)) / 255.0 for p in batch_paths]
            yield np.array(batch_images), np.array(batch_labels)


### Task 1: Create the list "all_image_paths" containing paths of all files for both folders, class_0_non_agri and class_1_agri, in the base directory.


In [2]:
all_image_paths = []
all_labels = []

label_non_agri = 0
label_agri = 1

for fname in os.listdir(dir_non_agri):
    all_image_paths.append(os.path.join(dir_non_agri, fname))
    all_labels.append(label_non_agri)

for fname in os.listdir(dir_agri):
    all_image_paths.append(os.path.join(dir_agri, fname))
    all_labels.append(label_agri)

print(f"Total image paths: {len(all_image_paths)}")
print(f"Total labels: {len(all_labels)}")


Total image paths: 5843
Total labels: 5843


### Task 2: Create a temporary list "temp" by binding the image paths and labels using the zip function. Next, randomly select and print 5 image paths and their corresponding labels.


In [3]:
temp = list(zip(all_image_paths, all_labels))
np.random.seed(42)
np.random.shuffle(temp)
all_image_paths, all_labels = zip(*temp)

print("First 5 paths and labels (after shuffling):", list(zip(all_image_paths[:5], all_labels[:5])))


First 5 paths and labels (after shuffling): [('./images_dataSAT/class_0_non_agri/tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_0.jpg', 0), ('./images_dataSAT/class_1_agri/tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_5878.jpg', 1), ('./images_dataSAT/class_0_non_agri/tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_10.jpg', 0), ('./images_dataSAT/class_1_agri/tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_5884.jpg', 1), ('./images_dataSAT/class_0_non_agri/tile_S2A_MSIL2A_20250409T105701_N0511_R094_T31UDQ_20250409T173716.SAFE_100.jpg', 0)]


### Task 3: Generate a batch of data (batch size = 8) by invoking the custom_data_generator function.


In [4]:
batch_size = 8
data_generator = custom_data_generator(image_paths=all_image_paths, 
                                       labels=all_labels, 
                                       batch_size=batch_size)

images, labels = next(data_generator)

print(f"Images batch shape: {images.shape}")
print(f"Labels batch shape: {labels.shape}")

plt.figure(figsize=(12, 6))
for i in range(batch_size):
    ax = plt.subplot(2, 4, i + 1)
    plt.imshow(images[i])
    plt.title(f"Label: {int(labels[i])}")
    plt.axis("off")
plt.tight_layout()
plt.show()


Images batch shape: (8, 64, 64, 3)
Labels batch shape: (8,)


<Figure size 1200x600 with 8 Axes>

### Task 4: Create validation data using a batch size of 8.


In [5]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir,
    labels='inferred',
    label_mode='int',
    validation_split=0.2,
    subset='validation',
    seed=1337,
    image_size=IMAGE_SIZE,
    batch_size=8
)


Found 5843 files belonging to 2 classes.
Using 1168 files for validation.
